# 부록 A — FlashAttention 실습 (Colab · A100)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Jake-Song/infer-opt/blob/main/notebooks/02b_flash_attention_a100_colab.ipynb)

*(배지는 이 노트북이 GitHub에 push된 뒤부터 동작합니다. 그 전에는 Colab의 파일 → 노트 업로드로 직접 올리세요.)*

2장 §7은 FlashAttention을 **개념과 대리 측정**으로만 다뤘습니다. 그 노트북이 도는 카드가
RTX 2080 SUPER(sm75)라 PyTorch의 진짜 `FLASH_ATTENTION` 백엔드를 못 쓰기 때문입니다.
그 백엔드는 **sm80 이상**을 요구합니다.

A100은 sm80입니다. 그래서 이 노트북에서는 2장이 못 한 것을 전부 합니다.

| | 2장 §7 (sm75) | 이 노트북 (A100, sm80) |
|---|---|---|
| `FLASH_ATTENTION` 백엔드 | 사용 불가 | **동작** |
| 진짜 FA2 커널 | 없음 | **`kernels-community/flash-attn2`** |
| online softmax 구현 | 생략 | **직접 작성하고 검증** |
| 최대 문맥 길이 | 8 GB에 막힘 | 40/80 GB까지 |
| flash-decoding (KV cache) | 없음 | **`flash_attn_with_kvcache`** |

### 이 노트북의 성격

**이 저장소를 clone하지 않습니다.** 필요한 것은 전부 아래에 정의되어 있고, Colab에 이미 있는
`torch`만 있으면 돌아갑니다. 2장을 안 읽었어도 따라올 수 있지만,
"왜 이게 중요한가"는 2장 §7과 §9에 있습니다.

**출력이 비어 있는 채로 커밋되어 있습니다.** 이 저장소의 다른 노트북은 측정값을 담아 커밋하지만,
이것만은 예외입니다 — 저자의 기계에 A100이 없어 실행할 수 없었기 때문입니다.
**여러분이 돌린 결과가 이 노트북의 첫 번째 숫자입니다.**

### 런타임 설정

Colab 메뉴에서 **런타임 → 런타임 유형 변경 → GPU → A100** 을 선택하세요 (Colab Pro 필요).
A100이 아니어도 대부분의 절은 돌아가지만, §4 이후의 핵심 비교는 sm80 이상이 필요합니다.

## §0. 런타임 확인

무엇 위에서 재는지부터 기록합니다. 이 노트북의 모든 숫자는 이 셀의 출력과 함께 읽어야 합니다.

In [ ]:
import subprocess, sys

import torch

print("python :", sys.version.split()[0])
print("torch  :", torch.__version__)
print("cuda   :", torch.version.cuda)

if not torch.cuda.is_available():
    raise SystemExit("GPU 런타임이 아닙니다. 런타임 → 런타임 유형 변경 → GPU 를 선택하세요.")

DEVICE = torch.device("cuda")
PROPS = torch.cuda.get_device_properties(DEVICE)
MAJOR, MINOR = torch.cuda.get_device_capability(DEVICE)
SM = MAJOR * 10 + MINOR

print()
print("gpu            :", PROPS.name)
print("compute cap    :", f"{MAJOR}.{MINOR}  (sm{SM})")
print("SMs            :", PROPS.multi_processor_count)
print("memory         :", f"{PROPS.total_memory / 1e9:.1f} GB")
print("bf16 supported :", torch.cuda.is_bf16_supported())

IS_AMPERE_PLUS = SM >= 80
IS_A100 = "A100" in PROPS.name
print()
if IS_A100:
    print("A100 확인. 이 노트북이 겨냥한 하드웨어입니다.")
elif IS_AMPERE_PLUS:
    print(f"A100은 아니지만 sm{SM} 이라 FLASH_ATTENTION 백엔드는 동작합니다. 절대 수치만 다릅니다.")
else:
    print(f"sm{SM} 입니다. §4 이후의 FlashAttention 백엔드 비교는 건너뜁니다 (sm80 이상 필요).")
    print("§1~§3 의 알고리즘 부분은 그대로 돌아갑니다.")

In [ ]:
DTYPE = torch.float16

def sync():
    torch.cuda.synchronize(DEVICE)


def timed_ms(fn, warmup: int = 5, iters: int = 20) -> float:
    """CUDA 이벤트로 재는 중앙값 (ms). 커널 실행은 비동기라 perf_counter 로는 못 잰다."""
    for _ in range(warmup):
        fn()
    sync()
    start = torch.cuda.Event(enable_timing=True)
    end = torch.cuda.Event(enable_timing=True)
    samples = []
    for _ in range(iters):
        start.record()
        fn()
        end.record()
        end.synchronize()
        samples.append(start.elapsed_time(end))
    samples.sort()
    return samples[len(samples) // 2]


def warmup_clocks(seconds: float = 3.0, size: int = 4096) -> None:
    """유휴 상태의 GPU는 클럭을 낮춰 둔다. 첫 측정이 몇 배 느리게 나오는 걸 막는다."""
    import time as _time
    left = torch.randn(size, size, device=DEVICE, dtype=DTYPE)
    right = torch.randn_like(left)
    deadline = _time.perf_counter() + seconds
    while _time.perf_counter() < deadline:
        for _ in range(10):
            left @ right
        sync()
    del left, right
    torch.cuda.empty_cache()


def free_gb() -> float:
    free, _ = torch.cuda.mem_get_info(DEVICE)
    return free / 1e9


warmup_clocks()
print(f"clocks warm, {free_gb():.1f} GB free")

## §1. 문제: score matrix

교과서적인 attention은 세 단계입니다.

$$S = \frac{QK^\top}{\sqrt{d}} \qquad P = \text{softmax}(S) \qquad O = PV$$

$Q$가 $T$개, $K$가 $T$개면 $S$는 $T \times T$입니다. head마다, batch마다요.
그리고 $S$와 $P$는 **HBM에 썼다가 다시 읽습니다.**

코드를 돌리기 전에 산수로 먼저 봅니다. 최적화는 측정에서 시작하지만,
**무엇을 잴지는 계산에서 나옵니다.**

In [ ]:
def score_matrix_bytes(batch: int, heads: int, seq_len: int, element_size: int = 2) -> int:
    """S 하나를 담는 데 필요한 바이트. P 까지 세면 두 배."""
    return batch * heads * seq_len * seq_len * element_size


def qkv_bytes(batch: int, heads: int, seq_len: int, head_dim: int, element_size: int = 2) -> int:
    """Q, K, V 를 합친 바이트 — 이쪽은 T 에 선형이다."""
    return 3 * batch * heads * seq_len * head_dim * element_size


HEADS, HEAD_DIM, BATCH = 32, 128, 1
print(f"batch={BATCH}, heads={HEADS}, head_dim={HEAD_DIM}, fp16 기준\n")
print(f"{'T':>7} {'Q+K+V':>12} {'S 하나':>14} {'S/QKV':>9} {'A100 80GB 대비':>16}")
for seq_len in (512, 1024, 2048, 4096, 8192, 16384, 32768):
    inputs = qkv_bytes(BATCH, HEADS, seq_len, HEAD_DIM)
    scores = score_matrix_bytes(BATCH, HEADS, seq_len)
    print(f"{seq_len:>7} {inputs / 1e6:>10.1f}MB {scores / 1e6:>12.1f}MB "
          f"{scores / inputs:>8.1f}x {scores / 80e9:>15.1%}")

print("\nQ+K+V 는 T 에 선형, S 는 T 의 제곱으로 자란다.")
print("T=16384 에서 입력은 400 MB 남짓인데 score matrix 하나가 A100 80GB 를 통째로 위협한다.")
print("그런데 S 는 softmax 를 통과한 직후 버려진다 — 최종 출력에는 O 만 남는다.")

### 여기서 읽어야 할 것

**입력은 선형, 중간값은 제곱입니다.** 그리고 그 제곱짜리 중간값은 **쓰자마자 버려집니다.**
$O = PV$를 계산하고 나면 $P$는 아무 쓸모가 없죠.

FlashAttention의 질문은 여기서 나옵니다. *버릴 것을 왜 HBM까지 보내는가?*

$S$의 한 블록은 몇 KB입니다. GPU의 SRAM(공유 메모리)에 들어가죠.
블록 하나를 만들어 SRAM에서 쓰고 버리면, HBM에는 최종 $O$만 쓰면 됩니다.

문제는 **softmax가 전역 연산**이라는 것입니다. $\text{softmax}(S)_i$는 그 행의
**모든** 원소를 알아야 정규화할 수 있습니다. 블록 하나만 보고는 계산할 수 없죠.

그 장벽을 넘는 것이 **online softmax**입니다. §2에서 직접 만듭니다.

## §2. Online softmax

수치적으로 안전한 softmax는 최댓값을 빼고 시작합니다.

$$\text{softmax}(x)_i = \frac{e^{x_i - m}}{\sum_j e^{x_j - m}}, \qquad m = \max_j x_j$$

$m$을 빼는 이유는 $e^{x}$의 오버플로 방지입니다. fp16에서 $e^{12}$면 이미 위험하죠.
그런데 $m$을 알려면 전체를 봐야 합니다. 이게 블록 처리를 막는 장벽입니다.

**해법: 지금까지 본 것의 최댓값으로 계산하고, 더 큰 값이 나오면 과거를 보정한다.**

블록 $j$까지 처리한 상태를 $(m_j, \ell_j, O_j)$라 합시다 — 각각 지금까지의 최댓값,
지수합, 출력 누적입니다. 새 블록 $S_{j+1}$이 오면:

$$m_{j+1} = \max(m_j,\; \text{rowmax}(S_{j+1}))$$
$$P_{j+1} = \exp(S_{j+1} - m_{j+1})$$
$$\ell_{j+1} = e^{m_j - m_{j+1}} \ell_j + \text{rowsum}(P_{j+1})$$
$$O_{j+1} = e^{m_j - m_{j+1}} O_j + P_{j+1} V_{j+1}$$

핵심은 **보정 계수** $e^{m_j - m_{j+1}}$입니다. 최댓값이 $m_j$에서 $m_{j+1}$로 올라갔으면
이전까지 누적한 값들은 전부 너무 큰 스케일로 계산된 상태입니다.
그걸 한 번 곱해서 새 기준으로 끌어내리는 거죠. 최댓값이 안 변했으면 계수는 1이고 아무 일도 없습니다.

마지막에 $O = O_N / \ell_N$으로 나누면 끝입니다. 먼저 softmax만 따로 검증해 봅니다.

In [ ]:
def online_softmax_sum(x: torch.Tensor, block: int = 64) -> tuple[torch.Tensor, torch.Tensor]:
    """x 를 블록으로 훑으며 (최댓값, 지수합) 을 누적한다. 전체를 한 번에 보지 않는다."""
    rows = x.shape[0]
    running_max = torch.full((rows, 1), float("-inf"), device=x.device, dtype=torch.float32)
    running_sum = torch.zeros((rows, 1), device=x.device, dtype=torch.float32)
    for start in range(0, x.shape[1], block):
        chunk = x[:, start : start + block].float()
        chunk_max = chunk.max(dim=-1, keepdim=True).values
        new_max = torch.maximum(running_max, chunk_max)
        rescale = torch.exp(running_max - new_max)           # 과거 보정 계수
        rescale = torch.nan_to_num(rescale, nan=0.0)          # 첫 블록: -inf - finite
        running_sum = rescale * running_sum + torch.exp(chunk - new_max).sum(dim=-1, keepdim=True)
        running_max = new_max
    return running_max, running_sum


torch.manual_seed(0)
probe = torch.randn(8, 1000, device=DEVICE, dtype=torch.float32) * 5

reference = torch.softmax(probe, dim=-1)
online_max, online_sum = online_softmax_sum(probe, block=64)
online = torch.exp(probe - online_max) / online_sum

print("online softmax vs torch.softmax")
print(f"  max |diff|     : {(online - reference).abs().max().item():.3e}")
print(f"  rows sum to 1  : {online.sum(-1).sub(1).abs().max().item():.3e}")
print(f"  블록 크기를 바꿔도 같은가: ", end="")
same = all(
    torch.allclose(
        torch.exp(probe - online_softmax_sum(probe, block=b)[0]) / online_softmax_sum(probe, block=b)[1],
        reference, atol=1e-6)
    for b in (1, 7, 64, 512, 1000)
)
print(same)

블록을 1개짜리로 쪼개든 통째로 보든 **같은 답**이 나옵니다. 그게 핵심입니다 —
online softmax는 근사가 아니라 **정확히 같은 값**을 순차적으로 계산하는 방법입니다.

이제 이걸 attention 전체에 끼워 넣습니다.

## §3. 타일링: FlashAttention의 루프를 직접 쓰기

§2의 재귀를 $O$ 누적까지 확장하면 그게 FlashAttention의 forward입니다.
아래는 그 루프를 **순수 PyTorch로** 옮긴 것입니다.

읽을 때 볼 것 두 가지:

1. **`scores` 변수의 크기.** `[rows, block]`이지 `[rows, T]`가 아닙니다.
   전체 score matrix는 어디에도 존재하지 않습니다.
2. **causal 처리.** query 블록보다 뒤에 있는 key 블록은 아예 **건너뜁니다**(`continue`).
   2장 §5에서 봤듯 계산한 뒤 마스크로 버리는 것과, 처음부터 안 하는 것은 다릅니다.
   삼각 마스크의 절반은 진짜로 계산되지 않습니다.

In [ ]:
import math


def flash_attention_reference(
    query: torch.Tensor, key: torch.Tensor, value: torch.Tensor,
    *, causal: bool = True, q_block: int = 128, kv_block: int = 128,
) -> torch.Tensor:
    """FlashAttention forward 를 파이썬 루프로 그대로 옮긴 것.

    shape: [batch, heads, seq, head_dim].  속도를 위한 코드가 아니라
    '전체 score matrix 없이도 정확히 같은 답이 나온다'를 보이기 위한 코드다.
    """
    batch, heads, q_len, head_dim = query.shape
    k_len = key.shape[2]
    scale = 1.0 / math.sqrt(head_dim)
    output = torch.zeros_like(query, dtype=torch.float32)

    for q_start in range(0, q_len, q_block):
        q_stop = min(q_start + q_block, q_len)
        q_tile = query[:, :, q_start:q_stop].float()                      # [B,H,qb,D]
        rows = q_stop - q_start

        running_max = torch.full((batch, heads, rows, 1), float("-inf"),
                                 device=query.device, dtype=torch.float32)
        running_sum = torch.zeros((batch, heads, rows, 1), device=query.device, dtype=torch.float32)
        acc = torch.zeros((batch, heads, rows, head_dim), device=query.device, dtype=torch.float32)

        for k_start in range(0, k_len, kv_block):
            k_stop = min(k_start + kv_block, k_len)
            # causal: 이 key 블록이 통째로 미래면 계산 자체를 하지 않는다.
            if causal and k_start > q_start + (k_len - q_len) + rows - 1:
                continue

            k_tile = key[:, :, k_start:k_stop].float()
            v_tile = value[:, :, k_start:k_stop].float()
            scores = (q_tile @ k_tile.transpose(-1, -2)) * scale          # [B,H,qb,kb]  <- 타일 하나뿐

            if causal:
                q_pos = torch.arange(q_start, q_stop, device=query.device) + (k_len - q_len)
                k_pos = torch.arange(k_start, k_stop, device=query.device)
                scores = scores.masked_fill(k_pos[None, :] > q_pos[:, None], float("-inf"))

            tile_max = scores.max(dim=-1, keepdim=True).values
            new_max = torch.maximum(running_max, tile_max)
            rescale = torch.nan_to_num(torch.exp(running_max - new_max), nan=0.0)

            probs = torch.exp(scores - new_max)
            probs = torch.nan_to_num(probs, nan=0.0)                       # 전부 -inf 인 행 보호
            running_sum = rescale * running_sum + probs.sum(dim=-1, keepdim=True)
            acc = rescale * acc + probs @ v_tile
            running_max = new_max

        output[:, :, q_start:q_stop] = acc / running_sum.clamp(min=1e-20)

    return output.to(query.dtype)

In [ ]:
import torch.nn.functional as F

torch.manual_seed(0)
B, H, T, D = 2, 4, 512, 64
q = torch.randn(B, H, T, D, device=DEVICE, dtype=torch.float32)
k = torch.randn(B, H, T, D, device=DEVICE, dtype=torch.float32)
v = torch.randn(B, H, T, D, device=DEVICE, dtype=torch.float32)

for causal in (True, False):
    expected = F.scaled_dot_product_attention(q, k, v, is_causal=causal)
    produced = flash_attention_reference(q, k, v, causal=causal, q_block=128, kv_block=128)
    print(f"causal={causal!s:<5} max |diff| vs SDPA : {(produced - expected).abs().max().item():.3e}")

print()
print("블록 크기를 바꿔도 답이 같은지:")
base = F.scaled_dot_product_attention(q, k, v, is_causal=True)
for qb, kb in ((32, 32), (64, 256), (128, 128), (512, 512)):
    diff = (flash_attention_reference(q, k, v, causal=True, q_block=qb, kv_block=kb) - base).abs().max()
    print(f"  q_block={qb:>4}, kv_block={kb:>4} : {diff.item():.3e}")

# decode 모양 (query 1개) 도 되는지 -- §7 로 이어진다
q1 = torch.randn(B, H, 1, D, device=DEVICE, dtype=torch.float32)
kc = torch.randn(B, H, T + 1, D, device=DEVICE, dtype=torch.float32)
vc = torch.randn(B, H, T + 1, D, device=DEVICE, dtype=torch.float32)
decode_expected = F.scaled_dot_product_attention(q1, kc, vc)   # query 1개 -> 마스크 불필요
decode_produced = flash_attention_reference(q1, kc, vc, causal=False, q_block=1, kv_block=128)
print(f"\ndecode (query 1개) max |diff| : {(decode_produced - decode_expected).abs().max().item():.3e}")

### 여기서 읽어야 할 것

**전체 score matrix를 만들지 않고도 완전히 같은 답이 나옵니다.** 블록 크기를 어떻게 잡든
오차는 부동소수점 수준입니다. FlashAttention은 근사가 아닙니다 —
이게 자주 오해받는 지점인데, 정확도를 팔아서 메모리를 사는 게 아닙니다.

**그리고 이 파이썬 버전은 느립니다.** 당연합니다. 블록마다 커널이 여러 개 뜨고,
`acc`와 `running_max`가 HBM을 왕복합니다. 진짜 FlashAttention이 빠른 이유는
알고리즘이 아니라 **이 루프 전체가 하나의 CUDA 커널 안에서, 누적값이 레지스터와 SRAM에
머문 채로** 돌기 때문입니다.

알고리즘은 지금 이해했고, 나머지는 구현입니다. §4부터는 그 구현을 씁니다.

## §4. PyTorch SDPA 백엔드 — A100에서 드디어 넷 다

`F.scaled_dot_product_attention`은 하나의 API 뒤에 여러 구현을 숨겨 두고
입력 모양과 하드웨어를 보고 고릅니다. `sdpa_kernel`로 강제할 수 있죠.

| 백엔드 | 정체 | 요구사항 |
|---|---|---|
| `MATH` | 융합 없는 참조 구현. $S$를 실체화 | 없음 |
| `EFFICIENT_ATTENTION` | cutlass 기반 memory-efficient attention | 넓음 (sm50+) |
| `FLASH_ATTENTION` | **FlashAttention-2** | **sm80+** |
| `CUDNN_ATTENTION` | cuDNN fused attention | sm80+ |

2장이 도는 sm75 카드에서는 아래 표의 3~4번째 줄이 `NOT available`로 나옵니다.
A100에서는 넷 다 떠야 합니다.

In [ ]:
import warnings

from torch.nn.attention import SDPBackend, sdpa_kernel

BACKENDS = [
    ("MATH", SDPBackend.MATH),
    ("EFFICIENT_ATTENTION", SDPBackend.EFFICIENT_ATTENTION),
    ("FLASH_ATTENTION", SDPBackend.FLASH_ATTENTION),
    ("CUDNN_ATTENTION", SDPBackend.CUDNN_ATTENTION),
]

probe_q = torch.randn(1, 8, 256, 64, device=DEVICE, dtype=DTYPE)
probe_k = torch.randn_like(probe_q)

AVAILABLE = {}
for name, backend in BACKENDS:
    try:
        with warnings.catch_warnings(), sdpa_kernel(backend):
            warnings.simplefilter("ignore")   # 거절 사유를 경고로 쏟아낸다; 아래 표가 요약한다
            F.scaled_dot_product_attention(probe_q, probe_k, probe_k, is_causal=True)
        sync()
        AVAILABLE[name] = True
    except Exception:
        AVAILABLE[name] = False
    print(f"  {name:<22} {'available' if AVAILABLE[name] else 'NOT available on this GPU'}")

del probe_q, probe_k
torch.cuda.empty_cache()

USABLE = [(n, b) for n, b in BACKENDS if AVAILABLE[n]]
print()
if AVAILABLE["FLASH_ATTENTION"]:
    print("FLASH_ATTENTION 사용 가능 — 이 노트북의 전제가 충족됐습니다.")
else:
    print("FLASH_ATTENTION 을 못 씁니다. sm80 이상(A100/H100 등) 런타임인지 확인하세요.")
    print("아래 절들은 사용 가능한 백엔드만으로 계속 진행됩니다.")

In [ ]:
warmup_clocks(2.0)

def backend_cost(backend, seq_len: int, batch: int = 1, heads: int = 32, head_dim: int = 128):
    """prefill attention 한 번의 peak 메모리(MB)와 시간(ms). OOM 이면 (None, None)."""
    try:
        query = torch.randn(batch, heads, seq_len, head_dim, device=DEVICE, dtype=DTYPE)
        key, value = torch.randn_like(query), torch.randn_like(query)
    except torch.cuda.OutOfMemoryError:
        torch.cuda.empty_cache()
        return None, None

    torch.cuda.empty_cache()
    torch.cuda.reset_peak_memory_stats()
    baseline = torch.cuda.memory_allocated()
    try:
        with warnings.catch_warnings(), sdpa_kernel(backend):
            warnings.simplefilter("ignore")
            out = F.scaled_dot_product_attention(query, key, value, is_causal=True)
            sync()
            peak = (torch.cuda.max_memory_allocated() - baseline) / 1e6
            del out
            ms = timed_ms(lambda: F.scaled_dot_product_attention(query, key, value, is_causal=True),
                          warmup=3, iters=10)
    except (torch.cuda.OutOfMemoryError, RuntimeError):
        peak = ms = None
    finally:
        del query, key, value
        torch.cuda.empty_cache()
    return peak, ms


SEQ_LENS = [512, 1024, 2048, 4096, 8192]
backend_results = {name: {"mb": [], "ms": []} for name, _ in USABLE}
for seq_len in SEQ_LENS:
    for name, backend in USABLE:
        mb, ms = backend_cost(backend, seq_len)
        backend_results[name]["mb"].append(mb)
        backend_results[name]["ms"].append(ms)

header = f"{'T':>7}" + "".join(f"{n.split('_')[0][:8]:>22}" for n, _ in USABLE)
print("peak attention memory (MB) / latency (ms), batch 1, 32 heads, head_dim 128, causal\n")
print(header)
for index, seq_len in enumerate(SEQ_LENS):
    row = f"{seq_len:>7}"
    for name, _ in USABLE:
        mb, ms = backend_results[name]["mb"][index], backend_results[name]["ms"][index]
        row += f"{'OOM':>22}" if mb is None else f"{mb:>12.1f} /{ms:>7.3f}"
    print(row)

In [ ]:
import matplotlib.pyplot as plt

PREFILL_COLOR, DECODE_COLOR, ACCENT_COLOR = "#2a78d6", "#eb6834", "#1baf7a"
SURFACE, INK, INK_SOFT, GRID = "#fcfcfb", "#0b0b0b", "#52514e", "#e4e3de"
BACKEND_COLOR = {
    "MATH": DECODE_COLOR, "EFFICIENT_ATTENTION": PREFILL_COLOR,
    "FLASH_ATTENTION": ACCENT_COLOR, "CUDNN_ATTENTION": "#7b4fd1",
}

plt.rcParams.update(
    {
        "figure.facecolor": SURFACE, "axes.facecolor": SURFACE, "savefig.facecolor": SURFACE,
        "axes.edgecolor": "#d8d7d2", "axes.labelcolor": INK_SOFT, "text.color": INK,
        "xtick.color": INK_SOFT, "ytick.color": INK_SOFT, "grid.color": GRID,
        "axes.grid": True, "grid.linewidth": 0.8, "axes.axisbelow": True,
        "axes.spines.top": False, "axes.spines.right": False,
        "font.size": 11, "figure.dpi": 110, "lines.linewidth": 2, "lines.markersize": 7,
        "axes.titlesize": 12, "axes.titlelocation": "left", "axes.titlepad": 12,
    }
)

figure, (left, right) = plt.subplots(1, 2, figsize=(12.4, 4.3))
for name, _ in USABLE:
    xs = [t for t, mb in zip(SEQ_LENS, backend_results[name]["mb"]) if mb is not None]
    mbs = [mb for mb in backend_results[name]["mb"] if mb is not None]
    mss = [ms for ms in backend_results[name]["ms"] if ms is not None]
    color = BACKEND_COLOR.get(name, INK_SOFT)
    if mbs:
        left.plot(xs, mbs, marker="o", color=color, label=name)
    if mss:
        right.plot(xs, mss, marker="o", color=color, label=name)

for axis, ylabel, title in (
    (left, "peak attention memory (MB)", r"Memory: $O(T^2)$ against $O(T)$"),
    (right, "ms per prefill attention", "Latency at the same arithmetic"),
):
    axis.set_xscale("log", base=2); axis.set_yscale("log")
    axis.set_xticks(SEQ_LENS); axis.set_xticklabels(SEQ_LENS)
    axis.set_xlabel("sequence length $T$"); axis.set_ylabel(ylabel)
    axis.set_title(title); axis.legend(frameon=False, fontsize=9)

figure.tight_layout()
plt.show()

### 여기서 읽어야 할 것

로그-로그 그래프에서 **기울기**를 보세요. `MATH`는 $T$가 2배일 때 메모리가 4배로
뜁니다(기울기 2). 나머지는 2배씩 늘어납니다(기울기 1). $O(T^2)$와 $O(T)$가
눈에 보이는 형태입니다.

`FLASH_ATTENTION`과 `EFFICIENT_ATTENTION`이 비슷해 보인다면 맞습니다 — 둘 다
score matrix를 실체화하지 않는 같은 계열입니다. 차이는 구현 품질과 지원 범위이고,
**FlashAttention-2는 그중 가장 빠른 쪽**입니다. 속도 그래프에서 그 차이가 보입니다.

그리고 `MATH`가 어디서 OOM으로 사라지는지 보세요. **그 지점이 융합 커널 없이
가능한 최대 문맥 길이**입니다. "이 모델은 문맥 32k를 지원한다"는 말의 상당 부분은
사실 "이 커널을 쓴다"는 뜻입니다.

## §5. 어디까지 갈 수 있는가

A100의 메모리를 끝까지 써 봅니다. `FLASH_ATTENTION` 하나만으로 문맥을 밀어 올려
어디서 멈추는지 보고, 같은 자리에서 `MATH`가 요구했을 메모리를 계산해 비교합니다.

계산은 실행보다 쌉니다 — 돌려보지 않아도 $T \times T$가 몇 바이트인지는 알 수 있으니까요.

In [ ]:
warmup_clocks(2.0)

LONG_SEQS = [4096, 8192, 16384, 32768, 65536]
flash_name = "FLASH_ATTENTION" if AVAILABLE.get("FLASH_ATTENTION") else (USABLE[-1][0] if USABLE else None)
flash_backend = dict(BACKENDS)[flash_name] if flash_name else None

print(f"tiled backend in use: {flash_name}")
print(f"free memory: {free_gb():.1f} GB\n")
print(f"{'T':>7} {'tiled peak':>12} {'tiled ms':>10} {'MATH would need':>18} {'ratio':>9}")

long_rows = []
for seq_len in LONG_SEQS:
    mb, ms = backend_cost(flash_backend, seq_len) if flash_backend else (None, None)
    hypothetical = score_matrix_bytes(1, 32, seq_len) / 1e6      # S 하나, P 는 세지도 않았다
    if mb is None:
        print(f"{seq_len:>7} {'OOM':>12} {'-':>10} {hypothetical:>16.0f}MB {'-':>9}")
    else:
        long_rows.append((seq_len, mb, ms, hypothetical))
        print(f"{seq_len:>7} {mb:>10.1f}MB {ms:>10.2f} {hypothetical:>16.0f}MB {hypothetical / mb:>8.0f}x")

print(f"\n'MATH would need' 는 score matrix 하나의 크기다. 실제 MATH 경로는 P 까지 들고 있으므로")
print(f"최소 그 두 배를 요구한다. A100 80GB 기준 T=32768 의 S 하나가 이미 {score_matrix_bytes(1, 32, 32768) / 1e9:.0f} GB다.")

## §6. 진짜 커널: Hugging Face `kernels`로 FlashAttention 2 불러오기

여기까지는 PyTorch가 감싸 준 FlashAttention이었습니다. 이제 **커널 자체**를 직접 부릅니다.

예전에는 `pip install flash-attn`으로 소스를 빌드해야 했고, Colab에서 20분씩 걸리거나
실패하곤 했습니다. Hugging Face의 [`kernels`](https://github.com/huggingface/kernels)는
**미리 빌드된 커널을 Hub에서 내려받습니다.** 컴파일이 없습니다.

우리가 쓸 저장소는 **`kernels-community/flash-attn2`** — Dao-AILab FlashAttention 2 빌드이고,
제공 함수는 다음과 같습니다.

- `flash_attn_func` — 일반 attention
- `flash_attn_varlen_func` — 패딩 없이 길이가 제각각인 배치 (연속 배칭의 기초)
- `flash_attn_with_kvcache` — **decode용**, KV 캐시를 직접 받는다 (§7)
- `flash_attn_qkvpacked_func`, `flash_attn_kvpacked_func` 등

### 주의할 점 두 가지

**1. `vllm-flash-attn3`가 아닙니다.** Hub의 vLLM 계열 저장소는
`kernels-community/vllm-flash-attn3` 하나이고 이름 그대로 **FlashAttention 3**입니다.
FA3는 **Hopper(H100, sm90) 전용**이라 A100에서는 돌지 않습니다.
A100에서 쓸 수 있는 FA2는 `kernels-community/flash-attn2`입니다. 아래 셀은 둘 다 시도하고
무엇이 로드됐는지 보고합니다.

**2. `get_kernel`은 torch 버전을 정확히 맞춰 봅니다.** 빌드 변형이
`torch29-cxx11-cu128-x86_64-linux` 같은 이름으로 올라가 있어서, Colab의 torch가
아직 빌드가 없는 버전이면 로드가 실패합니다. 그때는 에러 메시지가 어떤 변형이 있는지
전부 알려 줍니다 — 실패해도 §4의 `FLASH_ATTENTION` 백엔드가 같은 FA2이므로 비교는 계속됩니다.

In [ ]:
%pip install -q -U kernels

In [ ]:
import inspect

# (repo, 이 커널이 요구하는 최소 compute capability, 설명)
FA_CANDIDATES = [
    ("kernels-community/flash-attn2", 80, "FlashAttention 2 (Dao-AILab)"),
    ("kernels-community/vllm-flash-attn3", 90, "vLLM FlashAttention 3 (Hopper 전용)"),
]

FA = None
FA_REPO = None
try:
    from kernels import get_kernel
except ImportError as exc:
    print("kernels 를 불러오지 못했습니다:", exc)
    get_kernel = None

if get_kernel is not None:
    for repo, min_sm, note in FA_CANDIDATES:
        if SM < min_sm:
            print(f"건너뜀  {repo}")
            print(f"        {note} — sm{min_sm} 이상이 필요한데 이 GPU 는 sm{SM} 입니다.")
            print(f"        (내려받아도 실행할 수 없으므로 시도하지 않습니다.)")
            continue
        print(f"시도    {repo} — {note}")
        try:
            # version= 으로 고정할 수 있지만, 고정하면 그 리비전에 맞는 빌드가 없을 때
            # 실패합니다. 여기서는 최신을 씁니다. 재현성이 필요하면 version=N 을 주세요.
            candidate = get_kernel(repo)
        except Exception as exc:
            lines = [line for line in str(exc).strip().split("\n") if line.strip()]
            print(f"        실패: {type(exc).__name__}")
            for line in lines[:4]:
                print(f"          {line[:150]}")
            if len(lines) > 4:
                print(f"          ... ({len(lines) - 4} 줄 더)")
            continue
        print(f"        로드 성공")
        if FA is None:
            FA, FA_REPO = candidate, repo

print()
if FA is None:
    print("FA 커널을 로드하지 못했습니다. §4 의 FLASH_ATTENTION 백엔드로 계속합니다.")
    print("— 그 백엔드도 같은 FlashAttention 2 이므로 비교 자체는 성립합니다.")
    print("— 'Cannot find a build variant' 라면 Colab 의 torch 버전에 맞는 빌드가")
    print("  아직 없다는 뜻입니다. 위 목록에 어떤 torch 버전이 있는지 나옵니다.")
else:
    print(f"사용할 커널: {FA_REPO}")
    print("제공 함수:", [n for n in dir(FA) if n.startswith(("flash", "get_"))])

In [ ]:
# 커널마다 시그니처가 다르다. 문서를 믿지 말고 직접 확인한다.
if FA is not None:
    for name in ("flash_attn_func", "flash_attn_varlen_func", "flash_attn_with_kvcache"):
        fn = getattr(FA, name, None)
        if fn is None:
            print(f"{name}: 이 커널에는 없음\n")
            continue
        try:
            print(f"{name}{inspect.signature(fn)}\n")
        except (TypeError, ValueError):
            doc = (getattr(fn, "__doc__", "") or "").strip().split("\n")
            print(f"{name}: 시그니처를 읽을 수 없음. doc 첫 줄: {doc[0][:150] if doc else '(없음)'}\n")
else:
    print("커널이 없어 건너뜁니다.")

### 레이아웃이 다릅니다 — 여기서 가장 많이 틀립니다

PyTorch SDPA와 FlashAttention 커널은 **텐서 축 순서가 다릅니다.**

$$\text{SDPA: } [B,\; H,\; S,\; D] \qquad\qquad \text{FlashAttention: } [B,\; S,\; H,\; D]$$

FA 쪽이 **시퀀스가 앞**입니다. 2장 §8에서 head-major와 seq-major 레이아웃을 비교했던
바로 그 축 순서 문제이고, 여기서는 선택이 아니라 **커널이 요구하는 규약**입니다.

`.transpose(1, 2)`로 바꿔 넘겨야 하는데, transpose는 view라서 공짜처럼 보이지만
커널이 연속 메모리를 요구하면 내부에서 복사가 일어납니다. 아래에서 그 비용까지 포함해 잽니다.

FA2는 **GQA를 기본 지원**합니다. `k`/`v`의 head 수가 `q`보다 적으면 알아서 그룹으로 묶어 읽습니다 —
2장 §5의 `gqa_native`가 하던 일을 커널 수준에서 하는 셈입니다.

In [ ]:
warmup_clocks(2.0)

def sdpa_attention(q_bhsd, k_bhsd, v_bhsd, causal=True):
    with warnings.catch_warnings(), sdpa_kernel(flash_backend):
        warnings.simplefilter("ignore")
        return F.scaled_dot_product_attention(q_bhsd, k_bhsd, v_bhsd, is_causal=causal,
                                              enable_gqa=q_bhsd.shape[1] != k_bhsd.shape[1])


def fa_attention(q_bhsd, k_bhsd, v_bhsd, causal=True):
    """FA 커널은 [B, S, H, D] 를 받는다. 축을 바꿔 넣고 결과를 되돌린다."""
    q = q_bhsd.transpose(1, 2).contiguous()
    k = k_bhsd.transpose(1, 2).contiguous()
    v = v_bhsd.transpose(1, 2).contiguous()
    out = FA.flash_attn_func(q, k, v, causal=causal)
    if isinstance(out, (tuple, list)):
        out = out[0]
    return out.transpose(1, 2)


if FA is not None and hasattr(FA, "flash_attn_func"):
    torch.manual_seed(0)
    B, H, KVH, T, D = 2, 32, 8, 2048, 128          # GQA 32:8
    q = torch.randn(B, H, T, D, device=DEVICE, dtype=DTYPE)
    k = torch.randn(B, KVH, T, D, device=DEVICE, dtype=DTYPE)
    v = torch.randn_like(k)

    try:
        reference = sdpa_attention(q, k, v, causal=True)
        produced = fa_attention(q, k, v, causal=True)
        print(f"shape  sdpa {tuple(reference.shape)}  fa {tuple(produced.shape)}")
        print(f"max |diff| vs SDPA : {(produced - reference).abs().max().item():.3e}")
        print()
        sdpa_ms = timed_ms(lambda: sdpa_attention(q, k, v), warmup=5, iters=20)
        fa_ms = timed_ms(lambda: fa_attention(q, k, v), warmup=5, iters=20)
        print(f"  SDPA ({flash_name:<18}) : {sdpa_ms:7.3f} ms")
        print(f"  {FA_REPO:<26} : {fa_ms:7.3f} ms   (transpose+contiguous 포함)")
        print()
        print("두 숫자가 비슷하면 정상입니다 — 같은 FA2 커널을 서로 다른 껍데기로 부른 것이니까요.")
        print("FA 쪽이 조금 느리다면 레이아웃 변환 비용이고, 모델이 처음부터 [B,S,H,D] 로")
        print("텐서를 들고 있으면 사라지는 비용입니다.")
    except Exception as exc:
        print(f"호출 실패: {type(exc).__name__}: {exc}")
        print("\n위 §6 의 시그니처 출력을 보고 인자 이름을 맞춰 주세요 (커널 버전마다 다릅니다).")
    finally:
        del q, k, v
        torch.cuda.empty_cache()
else:
    print("FA 커널이 없어 건너뜁니다. §4 의 백엔드 비교가 같은 커널을 이미 측정했습니다.")

## §7. decode: `flash_attn_with_kvcache`

2장 §2가 보여준 것: **decode 시간의 60%가 attention**이고, 그 비용은 KV 캐시를 읽는 데 있습니다.
2장 §5는 decode에 causal mask가 필요 없다는 걸, §9는 캐시를 블록으로 쪼개는 법을 다뤘죠.

`flash_attn_with_kvcache`는 그 셋을 한 함수에 담고 있습니다.

- query 하나를 받고 (`seqlen_q = 1`)
- KV 캐시 텐서를 직접 받고 (`k_cache`, `v_cache`)
- 시퀀스마다 다른 길이를 받고 (`cache_seqlens`) — **패딩 없이**
- 새 K/V를 넘기면 캐시에 **제자리에서 써 주고** (`k`, `v`)
- 버전에 따라 `block_table`로 **PagedAttention 레이아웃**까지 받습니다

마지막 항목이 중요합니다. 2장 §9에서 우리 `PagedKVCache.read()`는 블록을 모아 연속 텐서로
**복사**했고, 그래서 contiguous 대비 30% 가까이 느렸습니다. 진짜 커널은 block table을 들고
블록이 놓인 자리에서 바로 읽습니다 — 그 복사가 없습니다.

아래 셀은 시그니처를 먼저 확인하고, 있는 인자만 써서 호출합니다.

In [ ]:
if FA is not None and hasattr(FA, "flash_attn_with_kvcache"):
    warmup_clocks(2.0)
    kv_fn = FA.flash_attn_with_kvcache
    try:
        params = set(inspect.signature(kv_fn).parameters)
    except (TypeError, ValueError):
        params = set()
    print("flash_attn_with_kvcache 가 받는 인자:", sorted(params) if params else "(introspect 불가)")
    print("  block_table 지원:", "block_table" in params)
    print("  cache_seqlens 지원:", "cache_seqlens" in params)
    print()

    torch.manual_seed(0)
    B, H, KVH, D = 8, 32, 8, 128
    MAX_CTX = 4096
    # FA 의 KV 캐시 레이아웃도 seq-major: [B, max_seqlen, n_kv_heads, head_dim]
    k_cache = torch.randn(B, MAX_CTX, KVH, D, device=DEVICE, dtype=DTYPE)
    v_cache = torch.randn_like(k_cache)
    query = torch.randn(B, 1, H, D, device=DEVICE, dtype=DTYPE)
    # 시퀀스마다 길이가 다르다 -- 패딩도, 마스크도 없다
    lengths = torch.randint(1024, MAX_CTX, (B,), device=DEVICE, dtype=torch.int32)
    print("cache_seqlens (시퀀스별 길이):", lengths.tolist())

    try:
        kwargs = {"causal": False}
        if "cache_seqlens" in params or not params:
            kwargs["cache_seqlens"] = lengths
        out = kv_fn(query, k_cache, v_cache, **kwargs)
        if isinstance(out, (tuple, list)):
            out = out[0]
        print("output shape:", tuple(out.shape))

        # 같은 답이 나오는지 SDPA 로 시퀀스마다 따로 확인한다
        worst = 0.0
        for i in range(B):
            length = int(lengths[i])
            q_i = query[i : i + 1].transpose(1, 2)                       # [1,H,1,D]
            k_i = k_cache[i : i + 1, :length].transpose(1, 2)            # [1,KVH,L,D]
            v_i = v_cache[i : i + 1, :length].transpose(1, 2)
            want = F.scaled_dot_product_attention(q_i, k_i, v_i, enable_gqa=True)
            worst = max(worst, (out[i : i + 1].transpose(1, 2) - want).abs().max().item())
        print(f"max |diff| vs per-sequence SDPA : {worst:.3e}")

        ms = timed_ms(lambda: kv_fn(query, k_cache, v_cache, **kwargs), warmup=10, iters=40)
        print(f"\nflash_attn_with_kvcache : {ms:.4f} ms  (batch {B}, 길이 제각각, 한 레이어)")
        print("패딩된 배치라면 가장 긴 시퀀스 길이만큼 전부 계산했어야 하는 일입니다.")
    except Exception as exc:
        print(f"\n호출 실패: {type(exc).__name__}: {exc}")
        print("위의 인자 목록을 보고 호출을 맞춰 보세요.")
    finally:
        del k_cache, v_cache, query
        torch.cuda.empty_cache()
else:
    print("flash_attn_with_kvcache 가 없어 건너뜁니다.")
    print("2장 §9 가 이 자리에서 순수 PyTorch gather 로 같은 일을 하고, 그 비용을 측정합니다.")

### 여기서 읽어야 할 것

**`cache_seqlens`가 이 절의 핵심입니다.** 시퀀스마다 길이가 다른 배치를,
패딩 없이, 마스크 없이 처리합니다.

패딩 방식이라면 길이가 1024인 시퀀스도 4096짜리 배치에 묶이는 순간 4096만큼 계산됩니다.
버려질 계산이죠. `cache_seqlens`는 커널에게 "이 행은 여기까지만"이라고 알려 주고,
커널은 거기서 멈춥니다.

**이것이 연속 배칭의 전제 조건입니다.** 2장 §9는 `PagedKVCache`가 시퀀스별 길이를
들 수 있지만 모델의 decode 경로가 배치를 한 덩어리로 전진시킨다는 한계를 남겼습니다.
그 한계를 푸는 쪽이 바로 이런 커널입니다 — 길이가 제각각인 배치를 그대로 먹을 수 있으니,
스케줄러가 끝난 요청을 빼고 새 요청을 끼워 넣어도 커널은 신경 쓰지 않습니다.

`block_table`까지 지원된다면 그 조각들이 물리적으로 흩어져 있어도 됩니다.
**그때 vLLM의 PagedAttention이 완성됩니다.**

## §8. 정리

### 이 노트북에서 확인한 것

1. **online softmax는 근사가 아니다.** 블록으로 쪼개 계산해도 정확히 같은 값이 나온다 (§2).
2. **타일링만으로 score matrix가 사라진다.** 알고리즘은 파이썬 60줄이면 쓸 수 있고,
   빠른 이유는 알고리즘이 아니라 그 루프가 한 커널 안에서 돈다는 데 있다 (§3).
3. **A100에서 `FLASH_ATTENTION` 백엔드가 동작한다.** 메모리는 $O(T)$, `MATH`는 $O(T^2)$ (§4).
4. **그 차이가 곧 최대 문맥 길이다.** 융합 커널 없이는 긴 문맥이 애초에 불가능하다 (§5).
5. **`kernels`로 빌드 없이 진짜 FA2 커널을 쓴다.** 레이아웃은 `[B, S, H, D]` (§6).
6. **`flash_attn_with_kvcache`가 decode·가변 길이·페이징을 한 함수에 담는다** (§7).

### FlashAttention을 한 문장으로

**연산량을 줄이지 않고 메모리 왕복을 줄인다.** attention이 memory-bound이기 때문에
그것만으로 배수의 속도가 나오고, $O(T^2)$ 중간값이 사라지기 때문에 긴 문맥이 가능해진다.

### 하드웨어별 정리

| | FA2 | FA3 |
|---|---|---|
| 최소 아키텍처 | sm80 (Ampere, A100) | sm90 (Hopper, H100) |
| PyTorch SDPA 백엔드 | `FLASH_ATTENTION` | 버전에 따라 |
| HF kernels 저장소 | `kernels-community/flash-attn2` | `kernels-community/flash-attn3`, `kernels-community/vllm-flash-attn3` |
| A100에서 | **동작** | 동작하지 않음 |

### 연습문제

1. **손익분기 문맥 길이를 구하라.** §4의 표에서 `MATH`와 `FLASH_ATTENTION`의 속도 차이는
   $T$가 작을 때 얼마인가? 두 백엔드가 비슷해지는(혹은 `MATH`가 이기는) $T$가 존재하는가?
   존재한다면 왜인가?
   *(힌트: 타일링에는 고정 비용이 있다. $T$가 아주 작으면 score matrix도 작다.)*

2. **head_dim의 영향을 재라.** FA2는 `head_dim`이 64, 128, 256일 때 성능 특성이 다르다.
   §4의 `backend_cost`를 `head_dim`에 대해 돌려서, `FLASH_ATTENTION`이 가장 효율적인
   지점을 찾아라. 그 값이 요즘 모델들이 head_dim 128을 쓰는 이유와 관계있는가?
   *(힌트: 타일이 SRAM에 들어가야 한다. head_dim이 커지면 한 타일에 담기는 행이 줄어든다.)*

3. **가변 길이의 이득을 정량화하라.** §7의 `cache_seqlens` 방식과, 같은 배치를
   가장 긴 길이로 패딩해서 `flash_attn_func`로 처리하는 방식의 시간을 비교하라.
   길이 분포가 고를 때와 한쪽으로 치우쳤을 때 이득이 어떻게 달라지는가?
   *(힌트: 패딩 낭비는 `1 - mean(lengths)/max(lengths)` 에 비례한다. 길이를
   `randint(1024, 4096)` 대신 `randint(64, 4096)` 으로 바꿔 보라.)*

### 되돌아가기

이 노트북은 2장 §7이 하드웨어 때문에 못 한 것을 채운 것입니다.
2장으로 돌아가면 이 커널이 **한 스텝 전체 안에서** 어느 정도 비중인지 (§2 연산 단위 분해),
그리고 KV 캐시 쪽에서 무엇이 더 남았는지 (§9 PagedAttention) 볼 수 있습니다.